# V2 kayitli sonuclar — salt okuma; completed fiziksel kabul degildir.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import json, sys
PACKAGE = Path.cwd().resolve()
PROJECT = next(p for p in (PACKAGE, *PACKAGE.parents) if (p / "src/reactor_pipeline/v2_students.py").is_file())
sys.path.insert(0, str(PROJECT / "src"))
from reactor_pipeline import v2_student_runner, v2_students
assert PACKAGE.name == "v2_stage_32_ogrenci_02_beavrs_v2_pilot_scope_v1"
info = v2_students.verify_package(PACKAGE, require_approval=False)
saved = v2_student_runner.saved_results(PACKAGE)
print("Grup:", saved["group"]["state"], "| completed fiziksel kabul degildir")
rows = []
for case in saved["cases"]:
    summary = case["summary"] or {}
    k = summary.get("k_effective", {})
    manifest_case = next(x for x in v2_students._read(PACKAGE / "manifest.json")["experiments"]
                         if x["experiment_id"] == case["experiment_id"])
    rows.append({"experiment_id": case["experiment_id"],
                 "scenario_id": case["scenario_id"], "state": case["state"],
                 "controls": manifest_case["controls"], "calculation": manifest_case["calculation"],
                 "k": k.get("mean"), "MC_standard_error": k.get("mc_standard_error"),
                 "duration_s": summary.get("duration", {}).get("value"),
                 "fields_npz": str(Path(case["attempt"]) / "fields.npz") if case["attempt"] else None,
                 "summary_json": str(Path(case["attempt"]) / "summary.json") if case["attempt"] else None,
                 "physical_acceptance": case["physical_acceptance"],
                 "statistical_convergence": case["statistical_convergence"]})
display(pd.DataFrame(rows))
for case in saved["cases"]:
    if case["state"] != "completed":
        continue
    with np.load(Path(case["attempt"]) / "fields.npz", allow_pickle=False) as fields:
        z = fields["z_edges_cm"]
        axial = fields["kappa_fission_mean"].sum(axis=(1, 2))
        k_history = fields["k_generation"].copy()
        entropy = fields["shannon_entropy"].copy()
    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    axes[0].step((z[:-1]+z[1:])/2, axial, where="mid")
    axes[0].set_title("kappa-fission eksenel")
    axes[1].plot(k_history); axes[1].set_title("k gecmisi")
    axes[2].plot(entropy); axes[2].set_title("Shannon entropisi")
    fig.suptitle(case["experiment_id"] + " — inceleme gerekli")
    display(fig)
    plt.close(fig)


Grup: not_started | completed fiziksel kabul degildir


,experiment_id,scenario_id,state,controls,calculation,k,MC_standard_error,duration_s,fields_npz,summary_json,physical_acceptance,statistical_convergence
0,V2-STAGE_32_OGRENCI_02-0001,27e0e31b6192d93a8ad95eda19609101842f066d83e326...,not_started,{'u235_enrichment_wt_percent': 2.8851606657697...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
1,V2-STAGE_32_OGRENCI_02-0002,4e96d99396c8aa67b3305459faa338fc8cc3b175c791cf...,not_started,{'u235_enrichment_wt_percent': 2.8952831401377...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
2,V2-STAGE_32_OGRENCI_02-0003,2752e5f89b2f852324739fc89de50764f32fea8925d5f0...,not_started,{'u235_enrichment_wt_percent': 1.7733692381488...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
3,V2-STAGE_32_OGRENCI_02-0004,525b09992b1d5df1d9cbe2897ab054167deaed7de86cd8...,not_started,{'u235_enrichment_wt_percent': 2.0089650838386...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
4,V2-STAGE_32_OGRENCI_02-0005,3da1dc68147fa3ab269e774101d2c705dee1d268b464bb...,not_started,{'u235_enrichment_wt_percent': 1.8785408541373...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
5,V2-STAGE_32_OGRENCI_02-0006,bba74fd353347e2d78419207cb18a04ec95aef71aff7e1...,not_started,{'u235_enrichment_wt_percent': 1.9593687325484...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
6,V2-STAGE_32_OGRENCI_02-0007,5cda7589517017c792b54b0806a2c2be63717bb4b1a5f7...,not_started,{'u235_enrichment_wt_percent': 1.7663820640293...,"{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required
7,V2-STAGE_32_OGRENCI_02-0008,d85d1dab70910a3801da4913f8505834ece9711ad70a74...,not_started,"{'u235_enrichment_wt_percent': 2.35, 'fuel_tem...","{'particles_per_generation': 80000, 'batches':...",None,None,None,None,None,not_assessed,review_required


## Teslim — hesap başlatmaz
Tamamlanan koşular için WSL terminalinde: `python scripts/v2_transfer.py export student_packages/v2_students_portable_v1/stage_32/ogrenci_02/v2_stage_32_ogrenci_02_beavrs_v2_pilot_scope_v1 deliveries/v2_stage_32_ogrenci_02_beavrs_v2_pilot_scope_v1.zip`. Durum `completed` olmadan teslim tam sayılmaz; varsayılan ZIP statepoint içermez.
